# Create / Update Data Agent

Creates or updates the **Azure Cost Agent** Data Agent and attaches the
`CostManagementOntology` as its data source.

**What this notebook does:**
1. Reads AI instructions from `data_agent_instructions.md` in Lakehouse Files
2. Looks up the `CostManagementOntology` in the workspace
3. Looks up any Power BI report for the dashboard link in the footer
4. Builds the full Data Agent definition (draft + published)
5. Creates or updates the Data Agent via the Fabric REST API

**Prerequisites:**
- Attach the Lakehouse that contains `data_agent_instructions.md` in Files
- Run `03_create_cost_ontology` first so the ontology exists

In [ ]:
# â”€â”€ Configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DATA_AGENT_NAME        = "Azure Cost Agent"
ONTOLOGY_NAME          = "CostManagementOntology"
INSTRUCTIONS_FILE      = "/lakehouse/default/Files/data_agent_instructions.md"
INSTRUCTIONS_REPO_FILE = "data_agent_instructions.md"  # fallback: same dir as notebook

In [ ]:
# â”€â”€ Get workspace context and Fabric token â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json, requests, time, base64, os, re
from notebookutils import mssparkutils

fabric_token = mssparkutils.credentials.getToken("pbi")
headers = {
    "Authorization": f"Bearer {fabric_token}",
    "Content-Type": "application/json",
}

# Resolve workspace ID from notebook context
workspace_id = spark.conf.get("trident.workspace.id")
print(f"Workspace ID: {workspace_id}")

In [ ]:
# â”€â”€ Read AI instructions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
instructions_text = None

# Try Lakehouse Files first
if os.path.isfile(INSTRUCTIONS_FILE):
    with open(INSTRUCTIONS_FILE, "r", encoding="utf-8") as f:
        instructions_text = f.read()
    print(f"Loaded instructions from Lakehouse Files ({len(instructions_text):,} chars)")
else:
    # Fallback: try relative to notebook
    fallback = os.path.join(os.path.dirname(os.getcwd()), INSTRUCTIONS_REPO_FILE)
    if os.path.isfile(fallback):
        with open(fallback, "r", encoding="utf-8") as f:
            instructions_text = f.read()
        print(f"Loaded instructions from repo fallback ({len(instructions_text):,} chars)")

if not instructions_text:
    raise FileNotFoundError(
        f"Could not find instructions at {INSTRUCTIONS_FILE} or repo fallback. "
        f"Upload data_agent_instructions.md to Lakehouse Files."
    )

print(f"Instructions preview: {instructions_text[:200]}...")

In [ ]:
# â”€â”€ Look up Ontology in the workspace â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
API = "https://api.fabric.microsoft.com/v1"

resp = requests.get(
    f"{API}/workspaces/{workspace_id}/items?type=Ontology",
    headers=headers,
)
resp.raise_for_status()
ontologies = [i for i in resp.json()["value"] if i["displayName"] == ONTOLOGY_NAME]
if not ontologies:
    raise RuntimeError(
        f"Ontology '{ONTOLOGY_NAME}' not found. Run 03_create_cost_ontology first."
    )
ontology_id = ontologies[0]["id"]
print(f"Ontology: {ONTOLOGY_NAME} ({ontology_id})")

In [ ]:
# â”€â”€ Look up Power BI report for dashboard link (optional) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
report_link = None

# First try .env (set by 02_deploy_dashboard)
env_path = '/lakehouse/default/Files/.env'
if os.path.isfile(env_path):
    with open(env_path, 'r') as ef:
        for line in ef:
            if line.strip().startswith('PBI_REPORT_LINK='):
                report_link = line.strip().split('=', 1)[1].strip()
                print(f'Dashboard link from .env: {report_link}')
                break

# Fallback: look up via Fabric API
if not report_link:
    try:
        resp = requests.get(
            f"{API}/workspaces/{workspace_id}/items?type=Report",
            headers=headers,
        )
        resp.raise_for_status()
        reports = resp.json().get("value", [])
        billing_report = next(
            (r for r in reports if "billing" in r["displayName"].lower() or "cost" in r["displayName"].lower()),
            reports[0] if reports else None,
        )
        if billing_report:
            report_link = (
                f"https://app.powerbi.com/groups/{workspace_id}"
                f"/reports/{billing_report['id']}?experience=power-bi"
            )
            print(f"Dashboard report from API: {billing_report['displayName']} ({billing_report['id']})")
    except Exception as e:
        print(f"Could not look up PBI report (non-fatal): {e}")

# Patch the instructions footer with the real dashboard link
if report_link:
    instructions_text = re.sub(
        r'https://app\.powerbi\.com/groups/[^)"\s]+',
        report_link,
        instructions_text,
    )
    print("Patched dashboard link in instructions")
else:
    print("No PBI report found â€” dashboard link left as-is")

In [ ]:
# â”€â”€ Build Data Agent definition â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Ontology entity elements (matches the schema from 03_create_cost_ontology)
ontology_elements = [
    {
        "id": "Subscription", "is_selected": True,
        "display_name": "Subscription", "type": "ontology.entity",
        "description": "subscriptionId,subscriptionName", "children": [],
    },
    {
        "id": "ResourceGroup", "is_selected": True,
        "display_name": "ResourceGroup", "type": "ontology.entity",
        "description": "resourceGroupId,resourceGroupName,subscriptionId", "children": [],
    },
    {
        "id": "Resource", "is_selected": True,
        "display_name": "Resource", "type": "ontology.entity",
        "description": "resourceId,resourceType,resourceGroupName,resourceGroupId,location", "children": [],
    },
    {
        "id": "CostRecord", "is_selected": True,
        "display_name": "CostRecord", "type": "ontology.entity",
        "description": "costRecordId,subscriptionId,resourceGroupId,resourceId,meterId,serviceId,usageDate,preTaxCost,quantity,unitOfMeasure,chargeType,location,Currency",
        "children": [],
    },
    {
        "id": "MeterCategory", "is_selected": True,
        "display_name": "MeterCategory", "type": "ontology.entity",
        "description": "meterId,meterCategory,meterSubCategory,meterName,unitOfMeasure", "children": [],
    },
    {
        "id": "Service", "is_selected": True,
        "display_name": "Service", "type": "ontology.entity",
        "description": "serviceId,serviceName,chargeType", "children": [],
    },
]

# Datasource config pointing to the ontology
datasource = {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/dataAgent/definition/dataSource/1.0.0/schema.json",
    "artifactId": ontology_id,
    "workspaceId": workspace_id,
    "dataSourceInstructions": None,
    "displayName": ONTOLOGY_NAME,
    "type": "ontology",
    "userDescription": None,
    "metadata": {},
    "elements": ontology_elements,
}

# Stage config with AI instructions
stage_config = {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/dataAgent/definition/stageConfiguration/1.0.0/schema.json",
    "aiInstructions": instructions_text,
}

# Top-level agent config
data_agent_config = {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/dataAgent/definition/dataAgent/2.1.0/schema.json",
}

# Publish info
publish_info = {
    "$schema": "https://developer.microsoft.com/json-schemas/fabric/item/dataAgent/definition/publishInfo/1.0.0/schema.json",
    "description": "",
}

def to_b64(obj):
    """JSON-serialize and base64-encode a dict."""
    return base64.b64encode(json.dumps(obj, ensure_ascii=False).encode("utf-8")).decode("ascii")

# Assemble all definition parts (draft + published use same content)
ontology_folder = f"ontology-{ONTOLOGY_NAME}"
definition_parts = [
    {"path": "Files/Config/data_agent.json",                              "payload": to_b64(data_agent_config), "payloadType": "InlineBase64"},
    {"path": "Files/Config/draft/stage_config.json",                      "payload": to_b64(stage_config),      "payloadType": "InlineBase64"},
    {"path": f"Files/Config/draft/{ontology_folder}/datasource.json",     "payload": to_b64(datasource),        "payloadType": "InlineBase64"},
    {"path": "Files/Config/publish_info.json",                            "payload": to_b64(publish_info),      "payloadType": "InlineBase64"},
    {"path": "Files/Config/published/stage_config.json",                  "payload": to_b64(stage_config),      "payloadType": "InlineBase64"},
    {"path": f"Files/Config/published/{ontology_folder}/datasource.json", "payload": to_b64(datasource),        "payloadType": "InlineBase64"},
]

print(f"Built {len(definition_parts)} definition parts")
for p in definition_parts:
    print(f"  {p['path']} ({len(p['payload'])} chars b64)")

In [ ]:
# â”€â”€ Helper: poll a Fabric long-running operation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def wait_for_operation(operation_id, label="Operation", max_wait=120):
    """Poll GET /operations/{id} until Succeeded or Failed."""
    elapsed = 0
    while elapsed < max_wait:
        r = requests.get(f"{API}/operations/{operation_id}", headers=headers)
        r.raise_for_status()
        state = r.json()
        status = state.get("status", "Unknown")
        if status == "Succeeded":
            print(f"  {label}: Succeeded")
            return state
        if status == "Failed":
            raise RuntimeError(f"{label} failed: {json.dumps(state.get('error'), indent=2)}")
        time.sleep(5)
        elapsed += 5
    raise TimeoutError(f"{label} timed out after {max_wait}s")

In [ ]:
# â”€â”€ Create or update the Data Agent â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

# Check if agent already exists
resp = requests.get(
    f"{API}/workspaces/{workspace_id}/items?type=DataAgent",
    headers=headers,
)
resp.raise_for_status()
existing = [i for i in resp.json()["value"] if i["displayName"] == DATA_AGENT_NAME]

if existing:
    # â”€â”€ Update existing Data Agent â”€â”€
    agent_id = existing[0]["id"]
    print(f"Updating existing Data Agent: {DATA_AGENT_NAME} ({agent_id})")

    update_body = {"definition": {"parts": definition_parts}}
    resp = requests.post(
        f"{API}/workspaces/{workspace_id}/items/{agent_id}/updateDefinition",
        headers=headers,
        json=update_body,
    )
    if resp.status_code == 200:
        print("  Updated successfully")
    elif resp.status_code == 202:
        op_id = resp.headers.get("x-ms-operation-id")
        wait_for_operation(op_id, "Update Data Agent")
    else:
        resp.raise_for_status()

else:
    # â”€â”€ Create new Data Agent â”€â”€
    print(f"Creating new Data Agent: {DATA_AGENT_NAME}")

    create_body = {
        "displayName": DATA_AGENT_NAME,
        "type": "DataAgent",
        "description": "Azure cost analysis agent backed by CostManagementOntology",
        "definition": {"parts": definition_parts},
    }
    resp = requests.post(
        f"{API}/workspaces/{workspace_id}/items",
        headers=headers,
        json=create_body,
    )
    if resp.status_code == 201:
        agent_id = resp.json()["id"]
        print(f"  Created ({agent_id})")
    elif resp.status_code == 202:
        op_id = resp.headers.get("x-ms-operation-id")
        wait_for_operation(op_id, "Create Data Agent")
        # Get the created item
        result = requests.get(f"{API}/operations/{op_id}/result", headers=headers)
        agent_id = result.json().get("id", "unknown")
        print(f"  Created ({agent_id})")
    else:
        resp.raise_for_status()

print(f"\nData Agent '{DATA_AGENT_NAME}' is ready.")
print(f"  Ontology attached: {ONTOLOGY_NAME} ({ontology_id})")
print(f"  Agent ID: {agent_id}")

In [ ]:
# â”€â”€ Verify: read back the definition and confirm ontology binding â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("Verifying Data Agent definition...")

resp = requests.post(
    f"{API}/workspaces/{workspace_id}/items/{agent_id}/getDefinition",
    headers=headers,
)
if resp.status_code == 202:
    op_id = resp.headers.get("x-ms-operation-id")
    wait_for_operation(op_id, "Get definition")
    result = requests.get(f"{API}/operations/{op_id}/result", headers=headers)
    result.raise_for_status()
    parts = result.json()["definition"]["parts"]
elif resp.status_code == 200:
    parts = resp.json()["definition"]["parts"]
else:
    resp.raise_for_status()

# Check datasource binding
for p in parts:
    if "datasource.json" in p["path"] and "draft" in p["path"]:
        ds = json.loads(base64.b64decode(p["payload"]).decode("utf-8"))
        bound_id = ds.get("artifactId", "")
        bound_name = ds.get("displayName", "")
        if bound_id == ontology_id:
            print(f"  Ontology binding verified: {bound_name} ({bound_id})")
        else:
            print(f"  WARNING: Expected ontology {ontology_id}, got {bound_id}")
        break

# Check instructions
for p in parts:
    if "stage_config.json" in p["path"] and "draft" in p["path"]:
        sc = json.loads(base64.b64decode(p["payload"]).decode("utf-8"))
        instr = sc.get("aiInstructions", "")
        print(f"  AI instructions: {len(instr):,} chars")
        break

print(f"\nDone. Open the Data Agent in Fabric portal to test it.")